In [ ]:
!pip install -q ultralytics opencv-python-headless tqdm

import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
import os

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/amitchakdhare/mumbai-coastal-road-video-sample"

video_path = None

for file in os.listdir(DATASET_PATH):
    if file.endswith((".mp4", ".avi", ".mov")):
        video_path = os.path.join(DATASET_PATH, file)
        break

if video_path is None:
    raise Exception("❌ No video found")

print("Using video:", video_path)

In [ ]:
def canny(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    return cv2.Canny(blur, 50, 150)

def region_of_interest(img):
    h, w = img.shape
    mask = np.zeros_like(img)

    polygon = np.array([[
        (0, h),
        (w, h),
        (w, int(h*0.6)),
        (0, int(h*0.6))
    ]], np.int32)

    cv2.fillPoly(mask, polygon, 255)
    return cv2.bitwise_and(img, mask)

def detect_lanes_and_fill(frame):
    edges = canny(frame)
    cropped = region_of_interest(edges)

    lines = cv2.HoughLinesP(cropped, 1, np.pi/180, 50,
                           minLineLength=40, maxLineGap=150)

    left_points = []
    right_points = []

    if lines is not None:
        for line in lines:
            x1,y1,x2,y2 = line[0]
            slope = (y2-y1)/(x2-x1+1e-6)

            if slope < -0.5:
                left_points.append((x1,y1))
                left_points.append((x2,y2))
            elif slope > 0.5:
                right_points.append((x1,y1))
                right_points.append((x2,y2))

    h, w, _ = frame.shape

    if left_points and right_points:
        left_points = np.array(left_points)
        right_points = np.array(right_points)

        left_bottom = tuple(left_points[left_points[:,1].argmax()])
        left_top = tuple(left_points[left_points[:,1].argmin()])

        right_bottom = tuple(right_points[right_points[:,1].argmax()])
        right_top = tuple(right_points[right_points[:,1].argmin()])

        lane_polygon = np.array([
            left_bottom,
            left_top,
            right_top,
            right_bottom
        ])

        overlay = frame.copy()
        cv2.fillPoly(overlay, [lane_polygon], (0,255,0))
        frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)

        lane_center = (left_bottom[0] + right_bottom[0]) // 2
    else:
        lane_center = w // 2

    return frame, lane_center

In [ ]:
model = YOLO('yolov8n.pt')

In [ ]:
output_path = '/kaggle/working/output.mp4'

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS) or 30
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(output_path,
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      fps,
                      (w,h))

video_speed_scale = 2.0
frame_skip = int(video_speed_scale)

prev_lane_center = None
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    if frame_count % (frame_skip + 1) != 0:
        continue

    # Lane
    frame, lane_center = detect_lanes_and_fill(frame)

    if prev_lane_center is None:
        prev_lane_center = lane_center
    else:
        lane_center = int(0.7 * prev_lane_center + 0.3 * lane_center)
        prev_lane_center = lane_center

    frame_center = w // 2
    deviation = lane_center - frame_center

    # YOLO
    results = model(frame, conf=0.5)[0]

    hazard = False

    for r in results.boxes.data.tolist():
        x1,y1,x2,y2,conf,cls = r
        name = model.names[int(cls)]

        if conf < 0.5:
            continue

        box_height = y2 - y1

        if y2 > h*0.6 and name in ["car","person","truck","bus"]:
            if box_height > h*0.25:
                hazard = True

        cv2.rectangle(frame,(int(x1),int(y1)),(int(x2),int(y2)),(255,0,0),2)
        cv2.putText(frame,name,(int(x1),int(y1)-5),
                    cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,0,0),1)

    # Decision
    if hazard:
        command = "STOP"
    else:
        if abs(deviation) < 30:
            command = "STRAIGHT"
        elif deviation > 0:
            command = "RIGHT"
        else:
            command = "LEFT"

    angle = deviation / w * 45

    # Draw
    cv2.line(frame,(frame_center,0),(frame_center,h),(0,0,255),2)
    cv2.circle(frame,(lane_center,int(h*0.9)),8,(0,255,255),-1)

    cv2.putText(frame,f"CMD: {command}",(20,40),
                cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,255),2)

    cv2.putText(frame,f"ANGLE: {angle:.2f}",(20,80),
                cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,255),2)

    out.write(frame)

cap.release()
out.release()

print("✅ DONE! Output saved at:", output_path)